# jepa-as-a-tool: one world model for three worlds, and the agent that uses it

An LLM agent (Muse Spark) plans pushes in a physical world. Its tool, `simulate`, runs a **world
model**: Meta's frozen **V-JEPA 2** encoder, a **predictor** that imagines the next latent after a
push, and a **translator** that reads positions out of a latent.

This notebook trains **one** world model on **three worlds at once** (PushT in 2D, a 3D tabletop,
3D billiards), then evaluates the agent in those three worlds and in a **fourth it never saw** (the
tabletop in another room). In every world, three ways to play the same tasks: **alone**, **with
our world model**, **with the real physics** (the upper bound).

| Step | What | Where | Time (estimate) |
|---|---|---|---|
| 1 | Agent alone and with the real physics, 4 worlds: starts now, runs in the background | API | all along |
| 2 | Random-push episodes of the 4 worlds | CPU | ~1.5 h |
| 3 | V-JEPA 2 latents | 2 GPUs | ~1 h |
| 4 | One translator for the 3 worlds | 1 GPU | ~15 min |
| 5 | One predictor for the 3 worlds | 1 GPU | ~3 h |
| 6 | Agent with the world model, 4 worlds, in the background | API + GPU | ~3 h |
| 7 | The world model in the room it never saw, against the real physics | 1 GPU | ~5 min |
| 8 | Wait for every episode, reports, `results.zip` | | until the last episode |

100 tasks per family, about 2,700 episodes in contributor mode. About 10 h in all: one Kaggle
session (12 h at most).

**Before running (Kaggle):**
1. Settings: Accelerator **GPU T4 x2**, Internet **on**.
2. Add-ons → Secrets → `META_API_KEY`, ticked for this notebook.
3. **Save Version → Save & Run All**: it keeps running with the browser closed.

No input is needed. If the session stops before the end, attach **this notebook's output** as an
input of a new version (Add Input → Notebooks → this notebook) and Save & Run All again: every step
skips what is done.

In [ ]:
import os
import sys

if os.path.isdir("/kaggle/working"):
    WORK = "/kaggle/working"
elif os.path.isdir("/content"):
    WORK = "/content"
else:
    WORK = os.getcwd()
REPO = f"{WORK}/jepa-as-a-tool"
if os.path.isdir(REPO):
    !git -C {REPO} pull
else:
    !git clone -b dev https://github.com/RahulCHANDER25/jepa-as-a-tool.git {REPO}
%cd {REPO}
!pip install -q -e .

# `pip install -e` only reaches new Python processes (the pipeline's jobs). This kernel started
# before it, so it finds `jat` through the source folder: same code as the jobs.
if f"{REPO}/src" not in sys.path:
    sys.path.insert(0, f"{REPO}/src")
!git log --oneline -1

## 0. Machine check and preflight

Rendering of the 2D and 3D worlds, then one job per GPU exactly like the pipeline's: CUDA, both 3D
worlds, rendering from several threads at once. If a line says FAILED, stop and send it to me.

In [ ]:
import subprocess

!nvidia-smi --query-gpu=name,memory.total --format=csv
!free -g | head -2
!df -h /tmp {WORK} | tail -2

CHECK = "from jat.worlds import get_world; f = get_world('billiards').render([150, 256, 400, 100]); g = get_world('pusht').render([256, 256, 0]); print('render ok', f.shape, g.shape)"


def renders(gl):
    env = {**os.environ, "MUJOCO_GL": gl}
    out = subprocess.run([sys.executable, "-c", CHECK], env=env, capture_output=True, text=True)
    print(gl, "->", (out.stdout or out.stderr).strip().splitlines()[-1])
    return out.returncode == 0


if renders("egl"):
    os.environ["MUJOCO_GL"] = "egl"
else:
    !apt-get -qq install -y libosmesa6 > /dev/null
    assert renders("osmesa"), "MuJoCo cannot render here"
    os.environ["MUJOCO_GL"] = "osmesa"
    os.environ["PYOPENGL_PLATFORM"] = "osmesa"
print("MUJOCO_GL =", os.environ["MUJOCO_GL"])

In [ ]:
import torch

from jat.pipeline import preflight

GPUS = torch.cuda.device_count()
preflight(GPUS)

## 1. API key and model

In [ ]:
def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret(name)
    except ImportError:
        pass
    try:
        from google.colab import userdata

        return userdata.get(name)
    except ImportError:
        pass
    from getpass import getpass

    return getpass(f"{name}: ")


os.environ["META_API_KEY"] = secret("META_API_KEY")
os.environ["META_API_BASE_URL"] = "https://api.meta.ai/v1"
# Contributor mode: same model, ~10x cheaper; Meta may train on these requests.
os.environ["META_MODEL"] = "muse-spark-1.3-contributor"
print("key loaded:", bool(os.environ["META_API_KEY"]), "| model:", os.environ["META_MODEL"])

## 2. The pipeline (and a previous session's progress, if attached)

Big data goes to `/tmp` (not saved); runs, results and logs go to `main-run/` in the output. Each
step's full log is in `main-run/logs/<step>.log`.

In [ ]:
import shutil
import zipfile
from pathlib import Path

from jat.pipeline import Pipeline

OUT = Path(WORK) / "main-run"
INPUTS = Path(WORK) / "inputs"
ROOTS = [Path("/kaggle/input"), INPUTS]


def earlier_sessions():
    """Folders with a previous session's runs/ and results/, the furthest along first."""
    found = {p.parents[2] for r in ROOTS for p in r.glob("**/runs/wm/translator.pt")}
    found = [f for f in found if (f / "results").is_dir() and f != OUT]
    return sorted(found, key=lambda f: -len(list(f.glob("results/*/episodes/*.json"))))


for archive in (p for r in ROOTS for p in r.glob("**/results*.zip")):
    if not (INPUTS / archive.stem).exists():
        zipfile.ZipFile(archive).extractall(INPUTS / archive.stem)
if (earlier := earlier_sessions()) and not (OUT / "runs/wm/translator.pt").exists():
    for part in ("runs", "results", "logs"):
        if (earlier[0] / part).exists():
            shutil.copytree(earlier[0] / part, OUT / part, dirs_exist_ok=True)
    print("continues", earlier[0])
else:
    print("a new run" if not (OUT / "runs").exists() else "continues this session's run")

Path("/tmp/jat-data").mkdir(exist_ok=True)
OUT.mkdir(exist_ok=True)
if not (OUT / "data").exists():
    (OUT / "data").symlink_to("/tmp/jat-data")  # latents stay out of the saved output

P = Pipeline(OUT, size="full", gpus=GPUS, provider="meta", eval_workers=8)
print(len(list(OUT.glob("results/*/episodes/*.json"))), "episodes so far")

### After a fix is pushed (harmless otherwise)

Pulls the fix into the running pipeline without losing anything; then run the failed cell again.

In [ ]:
!git -C {REPO} pull
import importlib

import jat.pipeline

importlib.reload(jat.pipeline)
P.__class__ = jat.pipeline.Pipeline
print("pipeline code:", jat.pipeline.__file__)

## Step 1: the agent alone and with the real physics (background)

No world model needed: these start now and run during training. `main-run/logs/eval-*.log` shows
their progress.

In [ ]:
P.stage_baseline_evals()

## Step 2: random-push episodes of the 4 worlds (CPU, ~1.5 h)

In [ ]:
P.stage_data()

## Step 3: V-JEPA 2 latents (2 GPUs, ~1 h)

In [ ]:
P.stage_encode()

## Step 4: one translator for the 3 worlds (~15 min)

A shared trunk on the V-JEPA 2 latents and one small head per world. Error on held-out frames:

In [ ]:
import json

P.stage_translator()
for world, errors in json.loads((P.wm / "translator_eval.json").read_text()).items():
    t = errors["test"]
    print(f"{world:>10}: {t['px_median']:.1f} px, {t['deg_median']:.1f} deg median")

## Step 5: one predictor for the 3 worlds (~3 h)

One world per training batch, in turn. The predictor is not told the world: the latent shows it.
Then its imagined positions against the real physics, after 1 to 4 pushes in a row:

In [ ]:
P.stage_predictor()
tests = json.loads((P.wm / "wm_eval.json").read_text())["test"]
for world, rows in tests.items():
    cells = "  ".join(f"{r['horizon']}: {r['px_median']:5.1f} px" for r in rows[1:5])
    print(f"{world:>10}  pushes ahead  {cells}")

## Step 6: the agent with the world model, in the 4 worlds (background)

In [ ]:
P.stage_wm_evals()

## Step 7: the world model in the room it never saw (~5 min)

The tabletop's physics, another table, light, color and camera. Read with the tabletop's head.

In [ ]:
P.stage_check_new_look()
rows = json.loads((P.wm / "wm_eval_tabletop-new-look.json").read_text())["test"]
for r in rows[:5]:
    print(f"{r['horizon']} pushes ahead: {r['px_median']:.1f} px, {r['deg_median']:.1f} deg median")

## Step 8: finish

Waits for every evaluation (failed episodes get one retry), writes `report.txt` per world, and packs
`results.zip`: runs, results and logs, no datasets.

In [ ]:
archive = P.stage_finish()
!ls -lh {archive}
for report in sorted(OUT.glob("results/*/report.txt")):
    print(f"\n===== {report.parent.name}\n{report.read_text()}")

## Send back

Download `main-run/results.zip` (Output tab) and send it.